# Create susceptibility charts

In this notebook we showcase how we can compute the multipactor thresholds measured during several tests and represent it on a susceptibility chart.

## Load data

In [ ]:
from functools import partial

from multipac_testbench import AveragedThresholdSet, TestCampaign, ThresholdSet
from multipac_testbench.data import config_path
from multipac_testbench.data.multipactor_tests import tests
from multipac_testbench.instruments import CurrentProbe, FieldProbe, ForwardPower
from multipac_testbench.util.multipactor_detectors import quantity_is_above_threshold

freqs = (120.0, 120.0, 120.0, 120.0, 120.0, 140.0, 140.0, 140.0, 140.0, 160.0, 160.0, 160.0, 160.0)
swrs = (5.0, 4.0, 3.0, 2.0, 1.0, 4.0, 3.0, 2.0, 1.0, 1.0, 2.0, 3.0, 4.0)

test_campaign =  TestCampaign.from_filepaths(
    tests,
    freqs,
    swrs,
    config_path,
    is_raw=True
)

## Determine thresholds

In [ ]:
current_multipactor_criterions = {'threshold': 12., 'minimum_number_of_points': 1}
current_multipac_detector = partial(quantity_is_above_threshold, **current_multipactor_criterions)

thresholds = test_campaign.determine_thresholds(
    current_multipac_detector,
    CurrentProbe,
    threshold_predicate=lambda t: t.sample_index > 300,
)

merged = test_campaign.determine_thresholds(
    current_multipac_detector,
    CurrentProbe,
    threshold_predicate=lambda t: t.sample_index > 300,
    threshold_reducer="any",
)
averaged = {test: AveragedThresholdSet.from_threshold_set(threshold_set) for test, threshold_set in merged.items()}


## Represent susceptibility plot

In [ ]:
_ = test_campaign.susceptibility_chart(merged, ForwardPower, figsize=(8, 8), xlim=(1e2, 1e3), ylim=(1e1, 1e4))

In [ ]:
axes, df = test_campaign.susceptibility_chart(thresholds, FieldProbe, figsize=(8, 8), xlim=(1e2, 1e3), ylim=(8e1, 2e3), color_according_to_swr=False)
axes.legend(loc='upper left', bbox_to_anchor=(1.05, 1.05))